# Lab 12B — Optimizacion: OPTIMIZE, ZORDER, Cache y Particionamiento

**Sesion 12 | Databricks Data Engineer Associate**  

**Runtime minimo:** DBR 13.3 LTS  
**Archivos fuente:** `ventas_detalle.csv`, `productos_catalogo.csv`

## Objetivos
- Medir el impacto de `OPTIMIZE` con `ZORDER BY` en tiempos de query
- Controlar el numero de particiones con `coalesce` vs `repartition`
- Usar `cache()` y `unpersist()` correctamente para reutilizar DataFrames
- Aplicar `VACUUM` para limpiar archivos obsoletos post-OPTIMIZE

## Setup previo
Subir los archivos al Volume:  
`/Volumes/dbassociate/default/vol_landing/sesion12/`

## Paso 0 — Verificacion del entorno

In [0]:
display(dbutils.fs.ls("/Volumes/dbassociate/default/vol_landing/"))

In [0]:
CATALOG       = "dbassociate"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD   = "gold"
VOLUME_PATH   = "/Volumes/dbassociate/default/vol_landing/sesion12"

SOURCE_VENTAS    = f"{VOLUME_PATH}/ventas_detalle.csv"
SOURCE_PRODUCTOS = f"{VOLUME_PATH}/productos_catalogo.csv"

print(f"Catalog : {CATALOG}")
print(f"Silver  : {SCHEMA_SILVER}")
print(f"Gold    : {SCHEMA_GOLD}")

## Paso 1 — Crear tabla Silver de ventas

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, to_date, round as spark_round, current_timestamp

schema_ventas = StructType([
    StructField("venta_id",        StringType(),  False),
    StructField("fecha_venta",     StringType(),  True),
    StructField("cliente_id",      StringType(),  True),
    StructField("producto_id",     StringType(),  True),
    StructField("cantidad",        IntegerType(), True),
    StructField("precio_unitario", DoubleType(),  True),
    StructField("descuento_pct",   DoubleType(),  True),
    StructField("region",          StringType(),  True),
    StructField("canal",           StringType(),  True),
    StructField("estado_pedido",   StringType(),  True),
    StructField("vendedor_id",     StringType(),  True),
])

df_raw = (
    spark.read
    .option("header", True)
    .schema(schema_ventas)
    .csv(SOURCE_VENTAS)
)

df_silver = (
    df_raw
    .withColumn("fecha_venta", to_date(col("fecha_venta"), "yyyy-MM-dd"))
    .withColumn("monto_bruto", spark_round(col("cantidad") * col("precio_unitario"), 2))
    .withColumn("monto_neto", spark_round(
        col("cantidad") * col("precio_unitario") * (1 - col("descuento_pct") / 100), 2
    ))
    .withColumn("ingestion_timestamp", current_timestamp())
    .filter(col("estado_pedido").isin("completado", "enviado", "entregado"))
    .dropDuplicates(["venta_id"])
)

(
    df_silver
    .write.format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.ventas_clean")
)

print(f"Tabla creada: {CATALOG}.{SCHEMA_SILVER}.ventas_clean")
display(df_silver.limit(5))

## Paso 2 — Medir tiempo de query ANTES de OPTIMIZE y ZORDER

Este es el baseline. El query filtra por `fecha_venta` y `region` — exactamente  
las columnas que luego se usaran en el ZORDER.

In [0]:
import time

query_bi = """
    SELECT region,
           DATE_TRUNC('month', fecha_venta) AS mes,
           COUNT(*)        AS num_ventas,
           SUM(monto_neto) AS total_neto,
           ROUND(AVG(monto_neto), 2) AS ticket_promedio
    FROM dbassociate.silver.ventas_clean
    WHERE fecha_venta BETWEEN '2023-01-01' AND '2023-06-30'
      AND region = 'Norte'
    GROUP BY region, DATE_TRUNC('month', fecha_venta)
    ORDER BY mes
"""

inicio = time.time()
spark.sql(query_bi).collect()
tiempo_antes = time.time() - inicio

print(f"Tiempo ANTES de OPTIMIZE: {tiempo_antes:.3f} segundos")
print("Tip: Spark UI > SQL > Files Read vs Files Pruned para este query")

## Paso 3 — OPTIMIZE con ZORDER BY

In [0]:
# OPTIMIZE: compacta los small files en archivos de ~1GB
# ZORDER BY (fecha_venta, region): ordena fisicamente los datos por esas columnas
# Efecto: los queries con WHERE fecha_venta AND region leera menos archivos (data skipping)
spark.sql("""
    OPTIMIZE dbassociate.silver.ventas_clean
    ZORDER BY (fecha_venta, region)
""")

print("OPTIMIZE con ZORDER completado.")

display(
    spark.sql("DESCRIBE HISTORY dbassociate.silver.ventas_clean")
    .select("version", "timestamp", "operation", "operationParameters")
    .limit(3)
)

## Paso 4 — Medir tiempo de query DESPUES de OPTIMIZE y ZORDER

In [0]:
inicio = time.time()
spark.sql(query_bi).collect()
tiempo_despues = time.time() - inicio

print(f"Tiempo ANTES  de OPTIMIZE : {tiempo_antes:.3f} segundos")
print(f"Tiempo DESPUES de OPTIMIZE: {tiempo_despues:.3f} segundos")

if tiempo_despues < tiempo_antes:
    mejora = (1 - tiempo_despues / tiempo_antes) * 100
    print(f"Mejora de rendimiento: {mejora:.1f}%")
else:
    print("Nota: con datos de laboratorio el dataset es pequenio.")
    print("En produccion con GBs la diferencia en 'Files Pruned' es notable en Spark UI.")

## Paso 5 — VACUUM: limpiar archivos obsoletos

`OPTIMIZE` genera una nueva version de la tabla con archivos compactados.  
Los archivos de versiones anteriores permanecen hasta que se ejecute `VACUUM`.

In [0]:
# VACUUM elimina archivos que ya no forman parte de ninguna version activa
# RETAIN 168 HOURS = 7 dias — el minimo recomendado para no romper Time Travel
spark.sql("VACUUM dbassociate.silver.ventas_clean RETAIN 168 HOURS")

print("VACUUM completado.")
print("Los archivos pre-OPTIMIZE fueron eliminados del storage.")
print("Advertencia: bajar la retencion por debajo de 7 dias rompe Time Travel.")

## Paso 6 — Particionamiento: coalesce vs repartition

Antes de escribir una tabla Gold pequenia, controlar el numero de archivos generados  
es critico. Sin control, Spark puede escribir 200 archivos de 1KB — small files problem.

In [0]:
# Aggregation con AQE activo — ver cuantas particiones genera realmente
from pyspark.sql.functions import spark_partition_id

df_gold = spark.sql("""
    SELECT
        region,
        DATE_TRUNC('month', fecha_venta) AS mes,
        canal,
        COUNT(*)        AS num_ventas,
        SUM(monto_neto) AS total_neto,
        ROUND(AVG(monto_neto), 2) AS ticket_promedio
    FROM dbassociate.silver.ventas_clean
    GROUP BY region, DATE_TRUNC('month', fecha_venta), canal
""")
num_partitions = df_gold.select(spark_partition_id().alias("pid")).distinct().count()
print(f"Particiones despues del aggregation (spark.sql.shuffle.partitions default): {num_partitions}")

display(df_gold)


In [0]:
# Aggregation con AQE activo — ver cuantas particiones genera realmente
from pyspark.sql.functions import spark_partition_id

df_gold = spark.sql("""
    SELECT
        region,
        DATE_TRUNC('month', fecha_venta) AS mes,
        canal,
        COUNT(*)        AS num_ventas,
        SUM(monto_neto) AS total_neto,
        ROUND(AVG(monto_neto), 2) AS ticket_promedio
    FROM dbassociate.silver.ventas_clean
    GROUP BY region, DATE_TRUNC('month', fecha_venta), canal
""")
df_gold = df_gold.repartition(20)

num_partitions = df_gold.select(spark_partition_id().alias("pid")).distinct().count()
print(f"Particiones despues del aggregation usando repartition: {num_partitions}")

display(df_gold)


In [0]:
# coalesce: combina particiones SIN shuffle — mas rapido, puede resultar desigual
# repartition: redistribuye CON shuffle — mas lento pero particiones uniformes
# Para una tabla Gold pequenia, coalesce(2) es suficiente

df_gold_2p = df_gold.coalesce(2)
num_partitions_2p = df_gold_2p.select(spark_partition_id().alias("pid")).distinct().count()

print(f"Particiones con coalesce(2): {num_partitions_2p}")

(
    df_gold_2p
    .write.format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.ventas_kpi_mensual")
)

print(f"Tabla Gold creada con 2 archivos: {CATALOG}.{SCHEMA_GOLD}.ventas_kpi_mensual")
display(df_gold_2p.orderBy("mes", "region"))

In [0]:
%sql
describe detail dbassociate.gold.ventas_kpi_mensual


### Diferencia clave: coalesce vs repartition

| Operacion | Shuffle | Resultado | Cuando usar |
|---|---|---|---|
| `coalesce(n)` | No | Puede haber particiones desiguales | Reducir particiones — tabla pequenia final |
| `repartition(n)` | Si | Particiones uniformes | Aumentar particiones o redistribuir para joins |
| `repartition(n, col)` | Si | Particiones por valor de columna | Pre-shuffle controlado para joins especificos |

## Paso 7 — Cache y Unpersist: reutilizar DataFrames

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema_productos = StructType([
    StructField("producto_id",     StringType(), False),
    StructField("nombre_producto", StringType(), True),
    StructField("categoria",       StringType(), True),
    StructField("subcategoria",    StringType(), True),
    StructField("costo_unitario",  DoubleType(), True),
    StructField("precio_lista",    DoubleType(), True),
    StructField("proveedor",       StringType(), True),
    StructField("activo",          StringType(), True),
])

df_productos = (
    spark.read
    .option("header", True)
    .schema(schema_productos)
    .csv(SOURCE_PRODUCTOS)
)

# Cachear: el catalogo de productos es pequenio y se usara en multiples operaciones
df_productos.cache()
n = df_productos.count()  # primera accion materializa el cache
print(f"Catalogo cacheado: {n} productos en memoria del executor.")

In [0]:
# Segunda operacion — usa el cache, no relanza la lectura del CSV
df_por_categoria = (
    df_productos
    .groupBy("categoria")
    .count()
    .orderBy("count", ascending=False)
)
display(df_por_categoria)

# Tercera operacion — tambien usa el cache
df_por_proveedor = (
    df_productos
    .groupBy("proveedor")
    .agg({"precio_lista": "avg", "costo_unitario": "avg"})
)
display(df_por_proveedor)

In [0]:
# Liberar el cache explicitamente — buena practica al final de su uso
df_productos.unpersist()
print("Cache liberado. La memoria del executor vuelve a estar disponible.")
print()
print("Antipatron: cachear el DataFrame de Bronze (millones de filas, se lee una sola vez)")
print("Patron correcto: cachear solo si el DataFrame se usa en MAS de una accion.")

## Paso 8 — Limpieza

In [0]:
spark.sql("DROP TABLE IF EXISTS dbassociate.silver.ventas_clean")
spark.sql("DROP TABLE IF EXISTS dbassociate.gold.ventas_kpi_mensual")
spark.catalog.clearCache()
print("Limpieza completada. Lab 12B finalizado.")

## Puntos clave del examen

1. `OPTIMIZE tabla ZORDER BY (col1, col2)` compacta archivos Y ordena fisicamente — son dos efectos distintos
2. `VACUUM` limpia los archivos obsoletos; nunca usar menos de `RETAIN 168 HOURS` (7 dias) en produccion
3. `coalesce(n)` no hace shuffle (rapido, puede ser desigual); `repartition(n)` hace shuffle (lento, uniforme)
4. `df.cache()` = `df.persist(MEMORY_AND_DISK)` — usar solo si el DataFrame se reutiliza en el mismo job
5. `ZORDER` sobre la columna mas frecuente en WHERE — no mas de 3-4 columnas o pierde efectividad